# Introducción a LangChain con RAG

**Caso.** Vas a crear un asistente para consultar el reglamento ficticio de un laboratorio universitario. Al preguntar por un préstamo, el asistente debe buscar los artículos pertinentes y citarlos en su respuesta.

**Qué vas a construir.** Primero preparas un índice de documentos. Después, para cada pregunta, recuperas fragmentos, los colocas en un prompt y generas una respuesta. RAG significa *retrieval-augmented generation*: generación aumentada por recuperación. Los fragmentos se agregan al contexto de la consulta, sin entrenar el modelo.

**Cómo trabajar.** Ejecuta las celdas en orden. Edita las celdas Markdown de respuesta con doble clic y guárdalas con Shift+Enter. Puedes trabajar en Jupyter Notebook, VS Code o Colab, con Python 3.10 o superior. La primera ejecución necesita internet para instalar paquetes y descargar dos modelos públicos. El notebook usa CPU si no hay GPU y funciona sin clave de API.

## 1. Prepara el entorno

`%pip` instala paquetes para el kernel activo. `langchain` y sus componentes aportan documentos, prompts, el índice y la composición del flujo. La integración `langchain-huggingface` conecta los embeddings. Transformers y PyTorch ejecutan los modelos localmente. Si actualizas librerías que ya estaban importadas, reinicia el kernel y continúa en la siguiente celda.

#### Dependencias del notebook

`%pip` instala paquetes en el entorno de Python del kernel. LangChain coordina el flujo, `langchain-text-splitters` corta el texto y `langchain-huggingface` conecta los embeddings. Sentence Transformers ejecuta el modelo de embeddings, mientras Transformers y PyTorch cargan y ejecutan Qwen. Los rangos permiten varias versiones compatibles y `==` fija una versión concreta.

In [1]:
%pip install "langchain>=1,<2" "langchain-text-splitters>=1,<2" "langchain-huggingface>=1,<2" "sentence-transformers>=5,<6" "transformers==4.56.2" "torch>=2.4"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Módulos y dispositivo

- **Python y PyTorch.** `sys` permite consultar la versión de Python. PyTorch comprueba si CUDA está disponible y permite usar CPU si hace falta.
- **LangChain.** `Document` guarda texto y metadatos, `ChatPromptTemplate` arma mensajes de chat y `RunnableLambda` convierte funciones de Python en etapas de la cadena. El divisor y el índice se encargan de fragmentar y buscar.
- **Hugging Face.** `HuggingFaceEmbeddings` conecta el índice con el modelo de embeddings. `AutoTokenizer` y `AutoModelForCausalLM` cargan el tokenizador y el generador.

In [2]:
import sys

import torch
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"  # Elige GPU CUDA si está disponible.
print("Python:", sys.version.split()[0], "| Dispositivo:", dispositivo)

C:\Users\EQUIPO\Desktop\ASIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.14.7 | Dispositivo: cpu


## 2. Carga los documentos

Un `Document` contiene texto en `page_content` y datos sobre su procedencia en `metadata`. Conservaremos el número de artículo para que puedas comprobar las citas del asistente. Los textos van incluidos en el notebook.

#### Artículos y metadatos

El diccionario `articulos` contiene el reglamento de ejemplo. La comprensión recorre sus pares de número y texto, y crea un `Document` por artículo. El texto se guarda en `page_content` y el identificador en `metadata`, para saber de dónde salió cada fragmento.

In [16]:
articulos = {  # Reglamento ficticio.
    "Art. 1": "Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. "
              "Las grabadoras pueden prestarse hasta por 24 horas. "
              "La devolución se realiza en el mostrador del laboratorio.",
    "Art. 2": "Reserva de equipo. Para reservar una cámara se requiere credencial vigente "
              "y enviar la solicitud al menos un día hábil antes del préstamo. "
              "La reserva queda confirmada cuando el personal responde por correo.",
    "Art. 3": "Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. "
              "El mostrador para devolución de equipo cierra a las 16:30.",
    "Art. 4": "Acceso a las salas. Los visitantes pueden entrar con autorización previa "
              "y acompañamiento de una persona responsable. Las salas tienen cupo de diez personas.",
    "Art. 5": "Reporte de fallas. Si detectas una falla en un equipo prestado, "
              "avisa al personal del laboratorio por correo. Incluye el número de inventario "
              "y una descripción breve de la falla.",
    "Art. 6": "Ingreso con alimentos y Bebidas. Queda completamente prohibido el ingreso con alimentos y bebidas, "
              "que no sean agua embotellada o en un termo personal . Los usuarios se haran responsables de un posible derramamiento"
              "hay sanciones aplicables.",
}
documentos = [  # Crea un documento por artículo.
    Document(page_content=texto, metadata={"articulo": numero})  # Conserva el identificador de la fuente.
    for numero, texto in articulos.items()
]
print(documentos[0])

page_content='Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse hasta por 24 horas. La devolución se realiza en el mostrador del laboratorio.' metadata={'articulo': 'Art. 1'}


## 3. Divide el texto en fragmentos

El buscador compara la pregunta con cada fragmento. Si el fragmento es muy corto, puede perderse la relación entre una condición y su excepción. Si es muy largo, arrastra detalles que no vienen al caso. El divisor intenta cortar por párrafos, líneas y palabras, en ese orden. Aquí el tamaño se mide en **caracteres**, y el solapamiento permite compartir texto entre fragmentos consecutivos del mismo documento.

#### División de documentos

`RecursiveCharacterTextSplitter` intenta separar por límites naturales como párrafos y espacios. Aquí el tamaño objetivo es de 180 caracteres y el solapamiento es de hasta 35. `split_documents` conserva los metadatos, y `enumerate` permite numerar los fragmentos para inspeccionarlos.

In [17]:
divisor = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=35)  # Define tamaño y solapamiento.
fragmentos = divisor.split_documents(documentos)  # Divide y conserva los metadatos.

for numero, fragmento in enumerate(fragmentos, start=1):
    fragmento.metadata["fragmento"] = numero  # Identifica cada fragmento.
    print(f'{numero:02} | {fragmento.metadata["articulo"]} | '
          f'{len(fragmento.page_content):3} caracteres | {fragmento.page_content}')

01 | Art. 1 | 178 caracteres | Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse hasta por 24 horas. La devolución se realiza en el mostrador del laboratorio.
02 | Art. 2 | 176 caracteres | Reserva de equipo. Para reservar una cámara se requiere credencial vigente y enviar la solicitud al menos un día hábil antes del préstamo. La reserva queda confirmada cuando el
03 | Art. 2 |  64 caracteres | reserva queda confirmada cuando el personal responde por correo.
04 | Art. 3 | 140 caracteres | Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. El mostrador para devolución de equipo cierra a las 16:30.
05 | Art. 4 | 157 caracteres | Acceso a las salas. Los visitantes pueden entrar con autorización previa y acompañamiento de una persona responsable. Las salas tienen cupo de diez personas.
06 | Art. 5 | 178 caracteres | Reporte de fallas. Si detectas una falla en un equipo prestado, avisa al personal del laborato

**Responde y registra lo que observas**

Cambia `chunk_size` a 100, ejecuta la celda y copia un fragmento que haya perdido contexto. Explica qué información perdió. Restablece 180 y vuelve a ejecutar antes de continuar.

**Respuesta.** 

01 | Art. 1 | 100 caracteres | Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse

Perdio información sobre cuantas horas pueden prestarse las grabadoras (son 24h)



## 4. Genera embeddings y crea el índice

Un *embedding* representa un texto como una lista de números. Usaremos un modelo multilingüe para que preguntas en español se puedan comparar con artículos en español. `InMemoryVectorStore` guarda los vectores y los documentos en RAM. Para cada búsqueda ordena los fragmentos por similitud de coseno con el vector de la pregunta.

#### Embeddings e índice vectorial

`HuggingFaceEmbeddings` usa MiniLM multilingüe para representar consultas y documentos con vectores comparables. `embed_query` permite inspeccionar la dimensión y algunas coordenadas de una consulta. `InMemoryVectorStore` usa ese mismo componente al agregar los fragmentos y mantiene el índice en RAM.

In [29]:
embeddings = HuggingFaceEmbeddings(  # Configura el modelo de embeddings.
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": dispositivo},  # Usa el dispositivo elegido.
    encode_kwargs={"normalize_embeddings": True},  # Normaliza los vectores.
)
vector = embeddings.embed_query("¿Cuánto dura el préstamo de una cámara?")  # Convierte la consulta a un vector.
print("Dimensión:", len(vector))
print("Primeras cinco coordenadas:", [round(valor, 4) for valor in vector[:5]])

indice = InMemoryVectorStore(embedding=embeddings)  # Crea el índice en RAM.
indice.add_documents(fragmentos)  # Indexa texto, vectores y metadatos.

Dimensión: 384
Primeras cinco coordenadas: [0.0327, 0.0095, -0.0464, -0.0778, 0.0476]


['85bc297d-1acb-43a6-9daf-829a04587d12',
 '68228034-7a8f-48e8-bfc9-4f09e3480136',
 '9a3c418f-bcd0-42cb-9398-4f51850dedce',
 '3a585d3d-9736-4455-b8fc-2111990ce0bf',
 '3a70d342-a248-492a-a700-5e48e57aa52f',
 'ddd47de1-2d08-4117-bde0-25e4dd349ac1',
 'a8ab8e25-f2df-483a-84e4-d06d4c6b87d3',
 '4ca558a5-1923-48ee-a1be-7c6c68222895']

## 5. Prueba la recuperación antes de generar

`k` indica cuántos fragmentos recuperar. La puntuación sirve para ordenar por cercanía, y **no** representa la probabilidad de que el fragmento contenga una respuesta. Un buscador de vecinos puede devolver texto incluso cuando la pregunta no está cubierta.

#### Recuperación por similitud

`similarity_search_with_score` busca los tres fragmentos más cercanos a la pregunta. Devuelve documentos y puntuaciones de similitud. Al imprimir artículo y texto puedes distinguir una coincidencia temática de un fragmento que realmente contiene la respuesta.

In [30]:
pregunta = "¿Dónde hay estacionamiento?"
for fragmento, similitud in indice.similarity_search_with_score(pregunta, k=3):  # Recupera hasta tres coincidencias.
    print(f'{similitud:.3f} | {fragmento.metadata["articulo"]} | {fragmento.page_content}')

0.188 | Art. 3 | Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. El mostrador para devolución de equipo cierra a las 16:30.
0.128 | Art. 2 | reserva queda confirmada cuando el personal responde por correo.
0.105 | Art. 2 | Reserva de equipo. Para reservar una cámara se requiere credencial vigente y enviar la solicitud al menos un día hábil antes del préstamo. La reserva queda confirmada cuando el


**Responde y registra lo que observas**

Identifica el fragmento que respalda una respuesta de «48 horas». Prueba «¿Dónde hay estacionamiento?» y explica por qué también devuelve fragmentos.

**Respuesta.** 

0.750 | Art. 1 | Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse hasta por 24 horas. La devolución se realiza en el mostrador del laboratorio.

Para: "¿Dónde hay estacionamiento?" sigo recibiendo respuestas porque el similarty_search_with_score lo que hace es buscar la similitudes entre los embeddings de la pregunta y de los articulos, busca vecinos mas cercanos a los de la pregunta, incluso si el indice vectorial es bajo como en este caso, responde a la pregunta.



## 6. Arma el prompt y carga el modelo generativo

`ChatPromptTemplate` prepara los mensajes con instrucciones, pregunta y contexto. El modelo generativo es **Qwen2.5-0.5B-Instruct**, un modelo pequeño para practicar. El tokenizador convierte los mensajes al formato de chat que espera Qwen y luego a tensores. `generate` añade tokens y se decodifica solamente la parte generada. El prompt muestra dos ejemplos de formato, con artículos ficticios que solo sirven para ilustrarlo. Tú verificarás si la respuesta cita los artículos recuperados.

#### Plantilla de chat y generación con Qwen

- **`ChatPromptTemplate`.** Combina una instrucción de sistema, dos ejemplos de respuesta y un mensaje que recibe `contexto` y `pregunta`. Los ejemplos muestran cómo citar y cómo indicar que falta información.
- **Modelo y tokenizador.** Transformers carga ambos componentes de Qwen. El modelo pasa al dispositivo elegido y entra en modo evaluación.
- **`generar`.** Traduce los roles de LangChain al formato de Qwen, aplica la plantilla de chat y mueve los tensores al dispositivo del modelo. `generate` devuelve la entrada seguida de los tokens nuevos. La función recorta la entrada y decodifica la respuesta.

In [31]:
plantilla = ChatPromptTemplate.from_messages([  # Define la plantilla de conversación.
    ("system", "Contesta la pregunta en una sola oración en español. "
               "Usa únicamente el fragmento que contiene el dato y cita su artículo. "
               "Si ningún fragmento contiene el dato, di: No hay información suficiente."),
    ("human", "Fragmentos: [Art. 90] La credencial se recoge en recepción. "
              "[Art. 91] El curso inicia en marzo.\nPregunta: ¿Dónde recojo mi credencial?"),
    ("ai", "La credencial se recoge en recepción. [Art. 90]"),  # Ejemplo de respuesta con evidencia.
    ("human", "Fragmentos: [Art. 90] La credencial se recoge en recepción."
              "\nPregunta: ¿Cuánto cuesta la credencial?"),
    ("ai", "No hay información suficiente."),  # Ejemplo de respuesta sin evidencia.
    ("human", "Fragmentos del reglamento:\n{contexto}\n\nPregunta: {pregunta}"),
])

modelo_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador = AutoTokenizer.from_pretrained(modelo_id)  # Carga el tokenizador de Qwen.
modelo = AutoModelForCausalLM.from_pretrained(  # Carga el modelo generativo.
    modelo_id, dtype=torch.float16 if dispositivo == "cuda" else torch.float32,
).to(dispositivo)
modelo.eval()  # Activa el modo de evaluación.

@torch.inference_mode()  # Desactiva el registro de gradientes.
def generar(prompt):
    roles = {"system": "system", "human": "user", "ai": "assistant"}  # Traduce los roles del chat.
    mensajes = [
        {"role": roles[mensaje.type], "content": mensaje.content}
        for mensaje in prompt.to_messages()
    ]
    entradas = tokenizador.apply_chat_template(  # Formatea y tokeniza los mensajes.
        mensajes, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(dispositivo)  # Mueve los tensores al dispositivo.
    salida = modelo.generate(**entradas, max_new_tokens=64, do_sample=False)  # Genera hasta 64 tokens nuevos.
    nuevos_tokens = salida[0, entradas["input_ids"].shape[1]:]  # Retira los tokens del prompt.
    return tokenizador.decode(nuevos_tokens, skip_special_tokens=True).strip()  # Convierte la continuación a texto.

## 7. Conecta recuperación y generación en LangChain

`as_retriever` expone la búsqueda del índice. Aquí empezamos con un solo fragmento para que puedas inspeccionar exactamente qué evidencia ve el modelo. `preparar` recupera los documentos, agrega su referencia al contexto y conserva los fragmentos para inspeccionarlos. El operador `|` conecta el prompt con el generador. `invoke` ejecuta el flujo con una entrada.

#### Cadena RAG

`as_retriever` convierte el índice en una etapa de recuperación y solicita un fragmento. `preparar` recupera fuentes y arma el contexto. `responder` pasa la pregunta y el contexto por la plantilla y el generador, y devuelve la respuesta junto con los documentos. `RunnableLambda` permite conectar ambas etapas con `|`. `mostrar_resultado` ejecuta el flujo e imprime la evidencia para poder contrastarla con la respuesta.

In [32]:
recuperador = indice.as_retriever(search_kwargs={"k": 1})  # Recupera un fragmento por consulta.
cadena_respuesta = plantilla | RunnableLambda(generar)  # Conecta la plantilla con Qwen.

def preparar(pregunta):
    fuentes = recuperador.invoke(pregunta)  # Busca los documentos pertinentes.
    contexto = "\n\n".join(  # Separa los fragmentos con una línea en blanco.
        f'[{doc.metadata["articulo"]}] {doc.page_content}' for doc in fuentes
    )
    return {"pregunta": pregunta, "contexto": contexto, "fuentes": fuentes}  # Conserva la consulta, el contexto y las fuentes.

def responder(datos):
    entrada = {"pregunta": datos["pregunta"], "contexto": datos["contexto"]}  # Selecciona los campos de la plantilla.
    return {"respuesta": cadena_respuesta.invoke(entrada), "fuentes": datos["fuentes"]}

rag = RunnableLambda(preparar) | RunnableLambda(responder)  # Encadena preparación y respuesta.

def mostrar_resultado(pregunta):
    resultado = rag.invoke(pregunta)  # Ejecuta la cadena completa.
    print("Pregunta:", pregunta)
    print("Respuesta:", resultado["respuesta"])
    print("Fragmentos recuperados:")
    for fuente in resultado["fuentes"]:  # Recorre las fuentes recuperadas.
        print(f'- [{fuente.metadata["articulo"]}] {fuente.page_content}')

mostrar_resultado("¿Cuánto tiempo puedo quedarme con una cámara prestada?")

Pregunta: ¿Cuánto tiempo puedo quedarme con una cámara prestada?
Respuesta: Las cámaras pueden prestarse hasta por 48 horas.
Fragmentos recuperados:
- [Art. 1] Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse hasta por 24 horas. La devolución se realiza en el mostrador del laboratorio.


## 8. Evalúa el RAG y modifícalo

Para cada caso revisa primero si se recuperó la evidencia adecuada y luego si la respuesta la usó correctamente. Si el dato está en los fragmentos pero la respuesta lo cambia, el fallo ocurrió durante la generación. Si el dato está en el reglamento pero no se recuperó, revisa la búsqueda.

#### Casos de evaluación

Cada par de `casos` reúne una consulta y un criterio de revisión. La cadena recibe únicamente la consulta. Después se imprime la referencia para comprobar si la respuesta coincide con la evidencia o, en el caso de estacionamiento, si indica que falta información.

In [33]:
casos = [  # Agrupa preguntas y criterios de revisión.
    ("¿A qué hora cierra el mostrador de devoluciones?", "16:30 [Art. 3]"),
    ("¿Qué datos debo poner en el correo para reportar una falla?", "Número de inventario y descripción [Art. 5]"),
    ("¿Cuánto cuesta el estacionamiento?", "Debe indicar que falta información"),
]
for consulta, referencia in casos:
    mostrar_resultado(consulta)  # Prueba el RAG con la consulta actual.
    print("Criterio de revisión:", referencia, "\n")

Pregunta: ¿A qué hora cierra el mostrador de devoluciones?
Respuesta: El mostrador para devolución de equipo cierra a las 16:30. [Art. 3] Horario del laboratorio.
Fragmentos recuperados:
- [Art. 3] Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. El mostrador para devolución de equipo cierra a las 16:30.
Criterio de revisión: 16:30 [Art. 3] 

Pregunta: ¿Qué datos debo poner en el correo para reportar una falla?
Respuesta: Reporte de fallas. Si detectas una falla en un equipo prestado, aviste al personal del laboratorio por correo. Incluye el número de inventario y una descripción breve de la falla.
Fragmentos recuperados:
- [Art. 5] Reporte de fallas. Si detectas una falla en un equipo prestado, avisa al personal del laboratorio por correo. Incluye el número de inventario y una descripción breve de la falla.
Criterio de revisión: Número de inventario y descripción [Art. 5] 

Pregunta: ¿Cuánto cuesta el estacionamiento?
Respuesta: No hay información sufi

**Responde y registra lo que observas**

Registra los tres resultados. Para cada uno marca si la evidencia aparece, si la respuesta coincide y si la cita existe en los fragmentos. En la pregunta sin respuesta esperada, indica si el modelo se abstuvo o inventó un dato.

**Respuesta.** 

Pregunta: ¿A qué hora cierra el mostrador de devoluciones?
Respuesta: El mostrador para devolución de equipo cierra a las 16:30. [Art. 3] Horario del laboratorio.
Fragmentos recuperados:
- [Art. 3] Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. El mostrador para devolución de equipo cierra a las 16:30.
Criterio de revisión: 16:30 [Art. 3]

**Veredicto:**

**La respuesta SI COINCIDE y la cita si exite en el fragmento**


Pregunta: ¿Qué datos debo poner en el correo para reportar una falla?
Respuesta: Reporte de fallas. Si detectas una falla en un equipo prestado, aviste al personal del laboratorio por correo. Incluye el número de inventario y una descripción breve de la falla.
Fragmentos recuperados:
- [Art. 5] Reporte de fallas. Si detectas una falla en un equipo prestado, avisa al personal del laboratorio por correo. Incluye el número de inventario y una descripción breve de la falla.
Criterio de revisión: Número de inventario y descripción [Art. 5]


**Veredicto:**

**La respuesta SI COINCIDE y la cita si exite en el fragmento**

Pregunta: ¿Cuánto cuesta el estacionamiento?
Respuesta: No hay información suficiente.
Fragmentos recuperados:
- [Art. 1] Préstamo de equipo. Las cámaras pueden prestarse hasta por 48 horas. Las grabadoras pueden prestarse hasta por 24 horas. La devolución se realiza en el mostrador del laboratorio.
Criterio de revisión: Debe indicar que falta información

**Veredicto:**

**El modelo se abstuvo correctamente**


#### Experimento con k

`k_prueba` controla cuántos fragmentos pide el recuperador. Cambia el valor a 1 y luego a 4 con la misma pregunta. La siguiente llamada a `mostrar_resultado` usa el recuperador recién configurado y permite observar el efecto en el contexto y la respuesta.

In [34]:
k_prueba = 4 # Cantidad de fragmentos solicitados.
consulta = " ¿Puedo ingresar con bebidas?"  # Mantiene fija la consulta.
recuperador = indice.as_retriever(search_kwargs={"k": k_prueba})  # Actualiza el recuperador.
mostrar_resultado(consulta)

Pregunta:  ¿Puedo ingresar con bebidas?
Respuesta: No puedo proporcionar información sobre esto.
Fragmentos recuperados:
- [Art. 6] Ingreso con alimentos y Bebidas. Queda completamente prohibido el ingreso con alimentos y bebidas, que no sean agua embotellada o en un termo personal . Los usuarios se haran
- [Art. 4] Acceso a las salas. Los visitantes pueden entrar con autorización previa y acompañamiento de una persona responsable. Las salas tienen cupo de diez personas.
- [Art. 3] Horario del laboratorio. El laboratorio abre de lunes a viernes de 09:00 a 17:00. El mostrador para devolución de equipo cierra a las 16:30.
- [Art. 2] Reserva de equipo. Para reservar una cámara se requiere credencial vigente y enviar la solicitud al menos un día hábil antes del préstamo. La reserva queda confirmada cuando el


**Responde y registra lo que observas**

Compara k=1 y k=4. Copia qué fragmentos cambiaron y explica cómo afectaron la respuesta. Agrega un Art. 6 al diccionario con una regla propia. Vuelve a ejecutar desde la creación de documentos y comprueba con una pregunta si el nuevo artículo aparece como fuente.

**Respuesta.** 

**Respuesta con k=1**: El mostrador para devolución de equipo cierra a las 16:30. [Art. 3] Horario del laboratorio.

**Respuesta con k=4**: El mostrador para devolución de equipo cierra a las 16:30.

La respuesta cambio brevemente, sobretodo notando que ocn k=1 si se incluyó el articulo citado, pero es cierto que agrega tal vez, malamente el nombre del articulo mientras que con k=4 no incluye ni el nombre ni el nuero de articulo. Quiero pensar que, con un solo k, el modelo cita bien porque en efecto, solo está obteniendo la informacion de un solo fragmento que cita a un solo articulo, sin embargo cuando se agregan mas fragmentos y por ende mas articulos, entiendo que el modelo razone que, lo mejor es no citar porque esta sacando la informacion de varios fragmentos, pero no deberia ser asi, pues solo uno de esos fragmentos incluye la informacion necesaria para respodner a la pregunta.

Agregué un nuevo articulo sobre el ingreso con bebidas y alimentos, con mi nueva pregunta el modelo no pudo responder y no citó el articulo.



## Referencias

- [LangChain, búsqueda y RAG](https://docs.langchain.com/oss/python/learn)
- [LangChain, InMemoryVectorStore](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore)
- [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)